# Fine-tune Llama 3.1 8B Instruct on BrainCode, then run the BBEH improvement benchmark

**Part 1** fine-tunes Llama 3.1 8B Instruct with LoRA (Unsloth) on validated BrainCode translations (`data/train.jsonl`: task → BrainCode) and saves the adapter to your Drive.

**Part 2** serves the base model and the fine-tuned adapter with vLLM, and runs the 50 BBEH-mini items in the **agentic BrainCode setup**. The model translates the task with tools for the spec, the retrieved glossary entries, glossary search, entry lookup and the checker. Then, in a fresh conversation, it solves the task from its BrainCode alone.

### Before you start
1. Upload the whole `finetune-llama` folder to **MyDrive** (so it is at `MyDrive/finetune-llama`).
2. **Runtime → Change runtime type → GPU**. An **A100** is best; an **L4** works. A **T4** (free tier) works too: the notebook switches to the 4-bit model, shorter training sequences, and a 16k serving context with capped tool outputs, so the agent sees less of the spec and glossary at once.
3. Optional: to push the adapter to the Hugging Face Hub, add your token as a Colab secret named `HF_TOKEN` (key icon in the left bar). It needs **write** permission.

Run the cells top to bottom. Part 2 starts with a runtime restart.

## Part 1: fine-tune

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
os.chdir('/content')     # never work inside the Drive mount: if it drops, every import fails (Errno 107)
from google.colab import drive
drive.mount('/content/drive')

FOLDER = '/content/drive/MyDrive/finetune-llama'   # results go here on Drive (change if elsewhere)
ZIP = '/content/drive/MyDrive/finetune-llama.zip'  # the folder as one zip: copies in seconds
WORK = '/content/finetune-llama'                   # a local copy to work from: fast, and immune to Drive drops
if not os.path.exists(f'{WORK}/data/train.jsonl'):  # after a session restart the local copy is still there
    if os.path.exists(ZIP):
        !unzip -q -o "{ZIP}" -d /content
    else:                                          # file by file from Drive: slow (minutes), shows progress
        assert os.path.exists(f'{FOLDER}/data/train.jsonl'), 'upload finetune-llama.zip (or the folder) to MyDrive'
        !rsync -a --info=progress2 --exclude runs --exclude outputs --exclude models "{FOLDER}/" "{WORK}/"
OUT = f'{FOLDER}/outputs'                          # adapter and zipped results are saved on Drive
os.makedirs(OUT, exist_ok=True)
os.makedirs(f'{FOLDER}/runs', exist_ok=True)       # benchmark runs are written straight to Drive (resumable)
if not os.path.islink(f'{WORK}/runs'):
    os.symlink(f'{FOLDER}/runs', f'{WORK}/runs')
%cd {WORK}

In [ ]:
%%capture
!pip install unsloth rank_bm25

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # less fragmentation (set before torch loads)
# plain downloads: the accelerated paths can leave a truncated tokenizer.json behind
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'
import torch

GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
LOAD_IN_4BIT = GPU_GB < 40          # A100: 16-bit LoRA; L4/T4: 4-bit QLoRA
# ungated copies of meta-llama/Llama-3.1-8B-Instruct (the 4-bit one is pre-quantized: 5.7 GB instead of 16 GB)
BASE_MODEL = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit' if LOAD_IN_4BIT else 'unsloth/Meta-Llama-3.1-8B-Instruct'
MAX_SEQ_LEN = 8192 if GPU_GB >= 40 else 4096   # A100: 8192 (~98% of examples); L4/T4: 4096 (~90%; 8192 runs out of memory)
INCLUDE_SPEC = False                # True: also put the compact spec (~10k tokens) in every training example
                                    #       (then set MAX_SEQ_LEN = 20000; training gets much slower)
EPOCHS = 2
LEARNING_RATE = 2e-4
LORA_R = 16
BATCH, GRAD_ACC = (2, 4) if GPU_GB >= 70 else (1, 8)   # 80 GB A100: 2 x 4; 40 GB A100, L4, T4: 1 x 8
ADAPTER_DIR = f'{OUT}/braincode-llama-lora'
print(f'GPU {GPU_GB:.0f} GB -> {BASE_MODEL}, max length {MAX_SEQ_LEN}, batch {BATCH} x {GRAD_ACC}')

### Training data
Every example is the same conversation the agentic runner uses for its translation step: the translator system prompt, then *"Translate this task into BrainCode:"* with the task, and the BrainCode as the answer. Training on that exact format means the fine-tuned model answers that prompt with BrainCode.

In [ ]:
import json, sys
sys.path.insert(0, WORK)
from run_braincode_agentic import TRANSLATE_SYSTEM

spec = open(f'{WORK}/braincode_kit/reference/language-spec.compact.md', encoding='utf-8').read()

def to_chat(row):
    system = TRANSLATE_SYSTEM + ('\n\n# BrainCode language specification\n\n' + spec if INCLUDE_SPEC else '')
    return {'messages': [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': 'Translate this task into BrainCode:\n\n' + row['item']},
        {'role': 'assistant', 'content': '```braincode\n' + row['braincode'] + '\n```'},
    ]}

def load(name):
    with open(f'{WORK}/data/{name}.jsonl', encoding='utf-8') as fh:
        return [json.loads(line) for line in fh if line.strip()]

train_rows, val_rows = load('train'), load('val')
print(json.load(open(f'{WORK}/data/manifest.json'))['by_dataset'])
print(len(train_rows), 'train /', len(val_rows), 'validation examples')

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

def load_base():
    return FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=LOAD_IN_4BIT, dtype=None)

free_gb = torch.cuda.mem_get_info()[0] / 1e9
assert free_gb > (8 if LOAD_IN_4BIT else 20), (
    f'only {free_gb:.1f} GB of GPU memory is free: an earlier model is still loaded. '
    'Runtime -> Restart session, then run the cells again from the Drive-mount cell (skip pip install).')
try:
    model, tokenizer = load_base()
except Exception as e:
    if 'expected value' not in str(e) and 'tokenizer' not in str(e).lower():
        raise
    # a truncated download ("expected value at line 1 column 1" from tokenizer.json): clear it and retry once
    import glob, shutil
    print('load failed:', e, '\n-> clearing the cached download and retrying once')
    for d in glob.glob(os.path.expanduser('~/.cache/huggingface/hub/models--' + BASE_MODEL.replace('/', '--') + '*')):
        shutil.rmtree(d, ignore_errors=True)
    model, tokenizer = load_base()
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=2026)
tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')

In [ ]:
from datasets import Dataset

def as_text(rows):
    texts = [tokenizer.apply_chat_template(to_chat(r)['messages'], tokenize=False)
             .removeprefix(tokenizer.bos_token) for r in rows]   # the tokenizer adds the BOS itself
    lengths = [len(tokenizer(t).input_ids) for t in texts]
    kept = [t for t, n in zip(texts, lengths) if n <= MAX_SEQ_LEN]
    print(f'{len(kept)}/{len(texts)} examples fit in {MAX_SEQ_LEN} tokens (longest {max(lengths)})')
    return Dataset.from_dict({'text': kept})

train_ds, val_ds = as_text(train_rows), as_text(val_rows)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field='text', max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC,
        num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, lr_scheduler_type='cosine', warmup_steps=10,
        weight_decay=0.01, optim='adamw_8bit', logging_steps=5, eval_strategy='epoch', save_strategy='no',
        bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
        output_dir='/content/checkpoints', report_to='none', seed=2026))
# learn only the BrainCode answers, not the prompts
trainer = train_on_responses_only(trainer,
    instruction_part='<|start_header_id|>user<|end_header_id|>\n\n',
    response_part='<|start_header_id|>assistant<|end_header_id|>\n\n')
stats = trainer.train()
print(stats)
print(trainer.evaluate())

In [ ]:
# Save the LoRA adapter to Drive (~170 MB). Part 2 serves it with vLLM.
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('saved', ADAPTER_DIR)

# Optional: push the adapter to the Hugging Face Hub (Colab secret HF_TOKEN, write permission)
PUSH_TO_HUB = False
HUB_REPO = 'your-username/braincode-llama-3.1-8b-lora'
if PUSH_TO_HUB:
    from google.colab import userdata
    model.push_to_hub(HUB_REPO, token=userdata.get('HF_TOKEN'), private=True)
    tokenizer.push_to_hub(HUB_REPO, token=userdata.get('HF_TOKEN'), private=True)

### Sanity check: translate a held-out item and run the BrainCode checker on it

In [ ]:
sys.path.insert(0, f'{WORK}/braincode_kit')
from rag.retrieve import Retriever, render_check

FastLanguageModel.for_inference(model)
row = val_rows[0]
msgs = to_chat(row)['messages'][:2]
inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=2048, temperature=0.6, top_p=0.9, do_sample=True)
answer = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
print(answer[:3000])

checker = Retriever(dense=False, glossary_path=f'{WORK}/braincode_kit/reference/glossary.jsonl',
                    index_dir=f'{WORK}/braincode_kit/rag/index', use_llm=False)
print(render_check(checker.check(answer, [])))

## Part 2: serve the model and run the benchmark

**First: Runtime → Restart session.** This frees the GPU memory from training and gives vLLM a clean environment. Files on Drive and in `/content` stay; Python variables are lost, so the next cell sets the paths again.

If your session disconnected after Part 1, start here: the adapter is on Drive.

In [ ]:
import os
os.chdir('/content')     # never work inside the Drive mount: if it drops, every import fails (Errno 107)
from google.colab import drive
drive.mount('/content/drive')

FOLDER = '/content/drive/MyDrive/finetune-llama'   # results go here on Drive (change if elsewhere)
ZIP = '/content/drive/MyDrive/finetune-llama.zip'  # the folder as one zip: copies in seconds
WORK = '/content/finetune-llama'                   # a local copy to work from: fast, and immune to Drive drops
if not os.path.exists(f'{WORK}/data/train.jsonl'):  # after a session restart the local copy is still there
    if os.path.exists(ZIP):
        !unzip -q -o "{ZIP}" -d /content
    else:                                          # file by file from Drive: slow (minutes), shows progress
        assert os.path.exists(f'{FOLDER}/data/train.jsonl'), 'upload finetune-llama.zip (or the folder) to MyDrive'
        !rsync -a --info=progress2 --exclude runs --exclude outputs --exclude models "{FOLDER}/" "{WORK}/"
OUT = f'{FOLDER}/outputs'                          # adapter and zipped results are saved on Drive
os.makedirs(OUT, exist_ok=True)
os.makedirs(f'{FOLDER}/runs', exist_ok=True)       # benchmark runs are written straight to Drive (resumable)
if not os.path.islink(f'{WORK}/runs'):
    os.symlink(f'{FOLDER}/runs', f'{WORK}/runs')
%cd {WORK}
import torch
ADAPTER_DIR = f'{FOLDER}/outputs/braincode-llama-lora'
assert os.path.exists(f'{ADAPTER_DIR}/adapter_config.json'), 'no adapter: run Part 1 first'
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
# context the agent works in (it reads the spec + retrieved glossary: ~20-35k tokens in full)
if GPU_GB >= 40:      # A100
    SERVE_MODEL, EXTRA, MAX_MODEL_LEN, RUN_FLAGS = 'unsloth/Meta-Llama-3.1-8B-Instruct', '', 65536, ''
elif GPU_GB >= 20:    # L4
    SERVE_MODEL, EXTRA, MAX_MODEL_LEN, RUN_FLAGS = 'unsloth/Meta-Llama-3.1-8B-Instruct', '', 32768, '--max-tool-chars 40000'
else:                 # T4: 16-bit weights do not fit; serve the 4-bit model (fp16, no bf16 on T4)
    SERVE_MODEL = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
    EXTRA, MAX_MODEL_LEN, RUN_FLAGS = '--quantization bitsandbytes --dtype half', 16384, '--max-tool-chars 20000 --workers 4'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'
print(f'GPU {GPU_GB:.0f} GB -> serving {SERVE_MODEL} with {MAX_MODEL_LEN} tokens of context')

In [ ]:
%%capture
!pip install vllm fastembed rank_bm25

In [ ]:
# vLLM loads the base model and serves the fine-tuned adapter on top of it as `braincode-llama`,
# with Llama 3.1 tool calling.
import subprocess, time, urllib.request, json
!wget -q -O /content/tool_template.jinja https://raw.githubusercontent.com/vllm-project/vllm/main/examples/tool_chat_template_llama3.1_json.jinja

cmd = (f'vllm serve {SERVE_MODEL} --served-model-name base-llama {EXTRA} '
       f'--enable-lora --lora-modules braincode-llama={ADAPTER_DIR} --max-lora-rank 16 '
       f'--max-model-len {MAX_MODEL_LEN} --gpu-memory-utilization 0.90 '
       f'--enable-auto-tool-choice --tool-call-parser llama3_json --chat-template /content/tool_template.jinja '
       f'--port 8000')
server = subprocess.Popen(cmd, shell=True, stdout=open('/content/vllm.log', 'w'), stderr=subprocess.STDOUT)

for _ in range(180):                       # model download + load: usually 3-8 minutes
    try:
        models = json.loads(urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5).read())
        print('server up:', [m['id'] for m in models['data']])
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError('vLLM exited; see /content/vllm.log:\n' + open('/content/vllm.log').read()[-3000:])
        time.sleep(10)
else:
    print(open('/content/vllm.log').read()[-3000:])

### Smoke test: 2 items, 1 run (check the output before the full run)

In [ ]:
!python run_braincode_agentic.py --model braincode-llama --condition smoke_ft --runs 1 --limit 2 {RUN_FLAGS}

### Full run: the fine-tuned model, agentic BrainCode setup (50 items × 3 runs)
Resumable: if Colab disconnects, rerun the cell; finished runs are kept and errored runs are redone.

In [ ]:
!python run_braincode_agentic.py --model braincode-llama --condition braincode_agentic_ft --runs 3 {RUN_FLAGS}

### Optional control: the fine-tuned model without BrainCode
Checks that fine-tuning did not change plain reasoning (compare with the base model's baseline run).

In [ ]:
!python run_braincode_agentic.py --model braincode-llama --condition baseline_ft --mode baseline --runs 3 {RUN_FLAGS}

### Results: accuracy per condition, and a zip of all runs to bring back to the repository

In [ ]:
import glob, shutil
from collections import defaultdict
acc = defaultdict(lambda: [0, 0, 0])
for p in glob.glob(f'{FOLDER}/runs/*/*/r*/result.json'):
    r = json.load(open(p))
    a = acc[r['condition']]
    if r['status'] == 'ok':
        a[0] += r['correct']; a[1] += 1
    else:
        a[2] += 1
for c, (k, n, e) in sorted(acc.items()):
    print(f'{c:28} {k:3}/{n:3} correct = {k / max(n, 1):.1%}   errors {e}')
shutil.make_archive(f'{FOLDER}/outputs/runs', 'zip', f'{FOLDER}/runs')
print('zipped ->', f'{FOLDER}/outputs/runs.zip')